In [7]:
import os
os.chdir(r"C:\Users\DstMove\Desktop\claude\projects\sutlab")

import sys
work_tree = r"\.claude\worktrees\musing-euler"
work_tree = ""
sys.path.insert(0, r"C:\Users\DstMove\Desktop\claude\projects\sutlab" + work_tree)

from pathlib import Path
from sutlab.io import load_metadata_from_excel, load_sut_from_parquet, load_balancing_targets_from_excel
from sutlab.sut import set_balancing_id, set_balancing_targets

EXAMPLES = Path("data/examples")
YEARS = list(range(2008, 2020))

metadata = load_metadata_from_excel(
    EXAMPLES / "metadata" / "ta_columns.xlsx",
    EXAMPLES / "metadata" / "ta_classifications.xlsx",
)
sut = load_sut_from_parquet(
    id_values=YEARS,
    paths=[EXAMPLES / f"ta_l_{y}.parquet" for y in YEARS],
    metadata=metadata,
    price_basis="current_year",
)

# Load targets back
targets = load_balancing_targets_from_excel(
    id_values=YEARS,
    paths=[EXAMPLES / f"ta_targets_l_{year}.xlsx" for year in YEARS],
    tolerances_path = EXAMPLES / "ta_targets_tolerances.xlsx",
    metadata=metadata,
)

In [3]:
sut = set_balancing_targets(sut, targets)

In [9]:
sut.balancing_targets.supply

,aar,trans,brch,maal
0,2008,0100,000001,20345472
1,2008,0100,000002,25715560
2,2008,0100,000004,5426475
3,2008,0100,010000,61862881
4,2008,0100,020000,3459559
...,...,...,...,...
1459,2019,0100,950000,5133513
1460,2019,0100,960000,13862789
1461,2019,0100,970000,4768750
1462,2019,0700,,1169251815


In [33]:
# Set balancing id and targets for a specific year
sut_2019 = set_balancing_id(sut, 2019)


In [35]:
sut_2019.balancing_id

2019

In [ ]:
import os
os.chdir(r"C:\Users\DstMove\Desktop\claude\projects\sutlab")

import numpy as np
import pandas as pd
from pathlib import Path
from sutlab.io import load_metadata_from_excel, load_sut_from_parquet, load_balancing_targets_from_excel
from sutlab.sut import set_balancing_id, set_balancing_targets

EXAMPLES = Path("data/examples")
YEARS = list(range(2008, 2020))

# Load metadata and SUT
metadata = load_metadata_from_excel(
    EXAMPLES / "metadata" / "ta_columns.xlsx",
    EXAMPLES / "metadata" / "ta_classifications.xlsx",
)
sut = load_sut_from_parquet(
    id_values=YEARS,
    paths=[EXAMPLES / f"ta_l_{y}.parquet" for y in YEARS],
    metadata=metadata,
    price_basis="current_year",
)

# Generate targets from actual data with small noise
cols = metadata.columns

supply_targets = (
    sut.supply
    .groupby([cols.id, cols.transaction, cols.category], as_index=False)[cols.price_basic]
    .sum()
    .rename(columns={cols.price_basic: cols.target})
)
use_targets = (
    sut.use
    .groupby([cols.id, cols.transaction, cols.category], as_index=False)[cols.price_purchasers]
    .sum()
    .rename(columns={cols.price_purchasers: cols.target})
)
targets_df = pd.concat([supply_targets, use_targets], ignore_index=True)

rng = np.random.default_rng(seed=42)
noise = rng.uniform(0.97, 1.03, size=len(targets_df))
targets_df[cols.target] = (targets_df[cols.target] * noise).round(1)

# Write one file per year
for year in YEARS:
    year_df = targets_df[targets_df[cols.id] == year].drop(columns=cols.id)
    year_df.to_excel(EXAMPLES / f"ta_targets_l_{year}.xlsx", index=False)

# Load targets back
targets = load_balancing_targets_from_excel(
    id_values=YEARS,
    paths=[EXAMPLES / f"ta_targets_l_{year}.xlsx" for year in YEARS],
    metadata=metadata,
)

# Set balancing id and targets for a specific year
sut_2019 = set_balancing_id(sut, 2019)
sut_2019 = set_balancing_targets(sut_2019, targets)

print("balancing_id:", sut_2019.balancing_id)
print("supply targets for 2019:")
print(sut_2019.balancing_targets.supply[sut_2019.balancing_targets.supply[cols.id] == 2019])


In [13]:
supply_targets = (
    sut.supply
    .groupby([cols.id, cols.transaction, cols.category], dropna=False, as_index=False)[cols.price_basic]
    .sum()
    .rename(columns={cols.price_basic: cols.target})
)

use_targets = (
    sut.use
    .groupby([cols.id, cols.transaction, cols.category], dropna=False, as_index=False)[cols.price_purchasers]
    .sum()
    .rename(columns={cols.price_purchasers: cols.target})
)

targets = pd.concat([supply_targets, use_targets], ignore_index=True)

rng = np.random.default_rng(seed=42)
noise = rng.uniform(0.97, 1.03, size=len(targets))
targets[cols.target] = (targets[cols.target] * noise).round(0)
targets = targets.set_index("aar")

In [14]:
for year in YEARS :
    targets.loc[year].to_excel(EXAMPLES / f"ta_targets_l_{year}.xlsx", index=False)